In [1]:
import os
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px


In [2]:
# 1. data loading

# load data class
project_root = Path.cwd().parent  # assumes you're in /notebooks
sys.path.append(str(project_root))

from backend.classes import CORDIS_data, Project_data
horizon_data = CORDIS_data(parent_dir=project_root, enrich=False)

# does not contain any df similar to projects.csv
# --> load from csv instead
df = pd.read_csv(os.path.join(project_root,'data/processed/projects.csv'), on_bad_lines='warn')
# add useful project metrics
n_publications = horizon_data.data_publications.groupby('project_id').size().rename('n_publications')
df = pd.merge(df, n_publications, left_on='id', right_on='project_id', how='outer')
df['duration_months_remainder']= df.duration_months%12

# make scientific field columns better for printing
for column in ['field_class', 'field', 'sub_field', 'niche']: 
    df[column] = [df[column][i][2:-2].replace("', '", ", ") for i in df.index]

df.total_cost = df.total_cost.fillna('Unknown')
df.n_publications = df.n_publications.fillna(0)

In [3]:
# 2. specify which things to plot and how

metrics_list = [
    'total_cost',
    'ec_max_contribution',
    'total_cost_per_year',
    'ec_contribution_per_year',
    'n_institutions',
    'n_publications',
    'duration_days'
]
metrics_labels = [
    'Total cost [€]',
    'EU funding [€]',
    'Total cost per year [€]',
    'EU funding per year [€]',
    'Number of collaborating organizations',
    'Number of publications',
    'Project duration [days]'
]
metrics_labels = dict(zip(metrics_list,metrics_labels))


# Specify hovertemplate: formats specified information shown in hover textbox
hovertemplate = \
'<b>%{hovertext}</b>'+\
'<br>Research fields: %{customdata[0]}'+\
'<br>EU funding contribution: €%{customdata[1]}'+\
'<br>Total Cost:  €%{customdata[2]}'+\
'<br># collaborating organizations: %{customdata[3]}'+\
'<br># publications: %{customdata[4]}'+\
'<br>Duration: %{customdata[5]} years %{customdata[6]} months'+\
'<extra></extra>'

# only this data can be shown in hover boxes (!! be sure to also update hovertemplate when making changes)
hover_data_list = [
    'niche',
    'ec_max_contribution',
    'total_cost',
    'n_institutions',
    'n_publications',
    'duration_years',
    'duration_months_remainder'
]

# create buttons
buttons1 = [dict(method = "update",
                 args = [{'x': [df[metric]]},{'xaxis': {'title': metrics_labels[metric]}}],
                 label = metrics_labels[metric])  
                 for metric in metrics_list]
buttons2 = [dict(method = "update",
                 args = [{'y': [df[metric]]},{'yaxis': {'title': metrics_labels[metric]}}], 
                 label = metrics_labels[metric]) 
                 for metric in metrics_list]

log_buttons = [
    dict(
        label="Linear X, Linear Y",
        method="relayout",
        args=[{"xaxis.type": "linear", "yaxis.type": "linear"}]
    ),
    dict(
        label="Log X, Linear Y",
        method="relayout",
        args=[{"xaxis.type": "log", "yaxis.type": "linear"}]
    ),
    dict(
        label="Linear X, Log Y",
        method="relayout",
        args=[{"xaxis.type": "linear", "yaxis.type": "log"}]
    ),
    dict(
        label="Log X, Log Y",
        method="relayout",
        args=[{"xaxis.type": "log", "yaxis.type": "log"}]
    ),
]

updatemenus = [
    dict(active=0,
        buttons=buttons1,
        x=1.05,
        y=.93,
        xanchor='left',
        yanchor='top',
        direction= 'down',
        showactive= True),
    dict(active=1,
        buttons=buttons2,
        x=1.05,
        y=.73,
        xanchor='left',
        yanchor='top'),
    dict(
        buttons=log_buttons,
        direction="down",
        x=1.05,
        y=.50,
        xanchor='left',
        yanchor='top'
        )
    ]
# annotations next to buttons:
annotations=[
        dict(
            text="X-axis:",
            x=1.10,
            y=1,
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left"
        ),
        dict(
            text="Y-axis:",
            x=1.10,
            y=0.8,
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left"
        ),
        dict(
            text="Axis Scale:",
            x=1.13,
            y=0.55,
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left"
        ),
    ]

In [4]:
# 3. plotting
x_metric,y_metric = 'total_cost', 'ec_max_contribution'
fig = px.scatter(df, 
                x=x_metric, 
                y=y_metric,
                hover_name='title',
                title='Impact analysis',
                custom_data=hover_data_list,
                labels = metrics_labels
                )
fig.update_traces(
    hovertemplate = hovertemplate,
    mode='markers',
    marker=dict(size=7, opacity=0.7),    
)
fig.update_layout(updatemenus=updatemenus, annotations=annotations)
fig.show()
